# Error Handling: General Principles

Now that we've covered the basics, let's go over some general principles when it comes to error handling.

Like most things concerning codebase and program design, these "rules" are more like a set of guidelines.
There will be cases when you want to deviate from them - you should always use your own judgement, since ultimately _you_ are the one who best understands your codebase!

![](https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExd2d4azl3b2R3NDZmcm5lb3RiMnMycmU3d2wyOXZza3NsaDdjNGk0aiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/8gJ28HfjAkc9y/giphy.gif)

## Where should I raise exceptions in my code?

The most natural thing to ask is when is it appropriate to raise exceptions in your codebase.
This question is closely related to when you feel it is important to perform validation checks on inputs (or settings) that users provide to your code.
Remember: you are a user of your own code too!

The main guiding principles I tend to use are:
- "How easy is it for a user to get this wrong?"
- "How complicated / niche / fundamental is the problem?"
- "Does someone else already handle this for me?"

### How easy is it for a user to get this wrong?

The idea behind this principle is that it's very off-putting for a new user to encounter problems with the common tools in your codebase, without some explanation for what they're doing wrong.
As they get used to using your code, they won't need these guide-rails as much, but to do that they have to not be put off in the first place!

Typically "how easy is it for a user to get this wrong" pertains to the entry-points of your codebase.
By which, I mean the main functions / classes / modules that you expect a user to interact with when using your codebase.
If you are providing a script to someone, you will also typically have a series of checks (and appropriate exceptions) on the inputs to that script, to help the user run it correctly (and which is why libraries like `argparse` are so helpful when writing command-line interfaces).

As an example, let's pretend I have built a super-fast matrix-equation solver:

In [2]:
import numpy as np
from numpy.linalg import solve # this is just so the code runs, pretend this is an innovative method.

def tri_solver(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solve the equation
    
    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector.
    """
    return solve(A, b)

however my solver only works for upper-triangular matrices $A$ and column vectors $b$.

If this is one of the main selling points of my package, I want to make sure that a new user gets helpful messages if they try to use this in the wrong way.
As such, I would probably consider adding two checks to handle theses easily-missed provisos when using my method.

It is also normally worth mentioning any such cases in your docstrings!

In [ ]:
def tri_solver(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector.
    
    Note that the method will raise an error if $A$ is not upper triangular or $b$ is
    not provided as a column vector.
    """
    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")
    if b.size != b.shape[0]:
        raise RuntimeError(f"RHS vector $b$ must be a column vector (got shape {b.shape})")

    return solve(A, b)

In [10]:
# Non-upper triangular matrix
tri_solver(
    np.array([
        [1, 0, 2],
        [1, 1, 0],
        [0, 1, 1],
    ]),
    np.ones((3,))
)

RuntimeError: Input matrix $A$ is not upper-triangular

In [11]:
# Not a column vector
tri_solver(
    np.array(
        [
            [1, 0, 2],
            [0, 1, 0],
            [0, 0, 1],
        ]
    ),
    np.ones((1,3)),
)

RuntimeError: RHS vector $b$ must be a column vector (got shape (1, 3))

### How complicated / niche / fundamental is the problem?

Another thing to consider about when you want to raise errors is whether you have any particularly complex portions of your code where there are several things that could go wrong.
Or if you have any very specific edge-cases that cause issues for your codebase.

Continuing our example above; let's suppose that my method does not work on upper-triangular matrices whose diagonal consists of a sequence of (non-strictly) decreasing values, whose sum is less than the total sum of all the other elements in the matrix.
This is what I would deem a "niche" case that causes a fundamental problem for my method, and thus should be handled explicitly - since I don't expect even advanced users of my code to spot this immediately if it occurs in their work.
(In addition here, if these "matrices" are being read in from some large data source, I also don't expect my users to have manually looked through all their data and spotted the matrices that can't be used by eye)!

In [14]:
def tri_solver(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector.

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")
    
    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if np.all(diag_A == sorted(diag_A, reverse=True)) and diag_A.sum() < upper_tri_A.sum():
        raise RuntimeError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    if b.size != b.shape[0]:
        raise RuntimeError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    return solve(A, b)

In [15]:
# That niche error case
tri_solver(
    np.array(
        [
            [3, 0, 9],
            [0, 2, 0],
            [0, 0, 1],
        ]
    ),
    np.ones((3,)),
)

RuntimeError: $A$ cannot have a decreasing diagonal that sums to less than the sum of the non-diagonal elements.

Another, perhaps more illustrative example in the DeepSKA wilderness can be [found in `gains`](https://github.com/NSs-FLF-RHUL/gains/blob/cb7dccf16ca6d2067a52720c4b5350869fa0bde9/src/gains/utils/misc.py#L81-L95).

Here, we have a function that is designed to distribute a mesh across the number of available cores, however performance drastically decreases if the number of cores is not a power of 2.
As such, the code checks whether or not it has a suitable number of CPUs available - if it does then it constructs the appropriate mesh and returns it.
Otherwise, it `raise`s a `MeshError` (which is a [custom `Exception` type](./conventions.ipynb#custom-exceptions) defined by the `gains` package, that has a fixed error message).

### Does someone else already handle this for me?

Some important things to remember are that - whilst providing clear and helpful error messages to the user is important, filling your code with a lot of `raise` statements will make it look cluttered.
And as a general software engineering principle, you also want to avoid reinventing the wheel.

As such, you should avoid `raise` or handling exceptions that you know are already handled by other parties.
For example, our matrix solver will naturally fail if $A$ is provided as a singular matrix.
However, because part (or in this example, all of, but you get what I mean in general!) of our method uses `numpy.linalg.solve`, and this method _already checks_ if the input matrix is singular, we don't need to handle this ourselves.

In [17]:
# Singular matrix that would otherwise be fine for our method
tri_solver(
    np.zeros((3,3)),
    np.ones((3,)),
)

LinAlgError: Singular matrix

### If someone wants to break something, they will be able to

Finally, it's important to note that whilst you can attempt to cover as many bases as you can when it comes to error handling, if someone really wants to break your code, they will find a way.
To this end, try not to fall into the trap of "I must safeguard everything against potential issues" - it simply is not possible.
You have to - and should - assume some level of familiarity with your codebase on your user's side.

This is why - beyond circumstances governed by the principles above - you shouldn't feel the need to catch everything.
In particular if your codebase is quite large; and involves a lot of classes that you don't expect users to be interacting with (or only experienced users to be doing so), you shouldn't feel obliged to cover all bases.
You _should_ record any problem cases somewhere though - either in your documentation / docstrings, code comments (less recommended), or any examples that you share with other people.